# SocialChoice 02 - Choix Social Formel en Lean 4

**Navigation** : [<< 01-Arrow-Impossibility-Theorem.ipynb](01-Arrow-Impossibility-Theorem.ipynb) | [03-Voting-Methods.ipynb >>](03-Voting-Methods.ipynb) | [Index](../README.md)

**Autres notebooks** : [04-Computational-Aggregation-SAT-Z3.ipynb](04-Computational-Aggregation-SAT-Z3.ipynb)

**Kernel** : Python 3

---

## Introduction

Ce notebook explore la **theorie du choix social**, et plus particulierement les celebres theoremes d'impossibilite d'**Arrow** (1951), de **Sen** (1970), et le **theoreme de l'electeur median** (Black 1948, Downs 1957).

Ces theoremes demontrent des limitations fondamentales des systemes de vote et de decision collective. Leur formalisation en Lean permet de verifier rigoureusement les preuves et d'explorer les hypotheses.

### Contexte historique

| Annee | Resultat | Auteur | Impact |
|-------|----------|--------|--------|
| 1785 | Paradoxe de Condorcet | Condorcet | Cycles dans les preferences collectives |
| 1948 | Theoreme de l'electeur median | Black | Convergence vers le centre |
| 1951 | Theoreme d'impossibilite | Arrow | Prix Nobel 1972 |
| 1970 | Paradoxe liberal | Sen | Conflit liberte/efficacite |

### Objectifs d'apprentissage

1. Formaliser les preferences et ordres sociaux
2. Definir et prouver le theoreme d'Arrow
3. Explorer le theoreme de Sen
4. Comprendre le theoreme de l'electeur median

### Duree estimee : 80 minutes

---

## 1. Definitions de Base

### 1.1 Preferences

Une **preference** est une relation binaire sur un ensemble d'alternatives qui est :
- **Complete** : pour tout $x, y$, soit $x \succeq y$ soit $y \succeq x$
- **Transitive** : si $x \succeq y$ et $y \succeq z$, alors $x \succeq z$

In [ ]:
-- Definitions de base pour la theorie du choix social

-- Relation de preference faible : x R y signifie "x est au moins aussi bon que y"
structure Preference (A : Type) where
  R : A → A → Prop
  complete : ∀ x y, R x y ∨ R y x
  trans : ∀ x y z, R x y → R y z → R x z

-- Preference stricte derivee : x P y ssi x R y et non(y R x)
def Preference.strict {A : Type} (p : Preference A) (x y : A) : Prop :=
  p.R x y ∧ ¬ p.R y x

-- Indifference : x I y ssi x R y et y R x
def Preference.indiff {A : Type} (p : Preference A) (x y : A) : Prop :=
  p.R x y ∧ p.R y x

#check Preference
#check @Preference.strict
#check @Preference.indiff

### 1.2 Profil de preferences

Un **profil** est une fonction qui associe a chaque individu sa preference.

In [ ]:
-- Profil de preferences : chaque individu a une preference sur les alternatives
def Profile (I A : Type) := I → Preference A

-- Fonction de bien-etre social (SWF)
-- Agregue les preferences individuelles en une preference sociale
structure SocialWelfareFunction (I A : Type) where
  f : Profile I A → Preference A

#check @Profile
#check @SocialWelfareFunction

### 1.3 Helpers pour manipuler les preferences

Ces fonctions permettent de construire des profils specifiques pour les preuves.

In [ ]:
-- Mettre une alternative en tete (preferee a toutes les autres)
def makeTop {A : Type} [DecidableEq A] (p : Preference A) (a : A) : Preference A := {
  R := fun x y => if x = a then True else if y = a then False else p.R x y
  complete := by
    intro x y
    by_cases hx : x = a
    · simp [hx]
    · by_cases hy : y = a
      · simp [hx, hy]
      · simp [hx, hy]; exact p.complete x y
  trans := by
    intro x y z hxy hyz
    by_cases hx : x = a
    · simp [hx]
    · by_cases hy : y = a
      · simp [hx, hy] at hxy
      · by_cases hz : z = a
        · simp [hx, hy, hz] at hyz
        · simp [hx, hy, hz] at *
          exact p.trans x y z hxy hyz
}

-- Mettre une alternative en bas (moins preferee que toutes les autres)
def makeBot {A : Type} [DecidableEq A] (p : Preference A) (a : A) : Preference A := {
  R := fun x y => if y = a then True else if x = a then False else p.R x y
  complete := by
    intro x y
    by_cases hy : y = a
    · simp [hy]
    · by_cases hx : x = a
      · simp [hx, hy]
      · simp [hx, hy]; exact p.complete x y
  trans := by
    intro x y z hxy hyz
    by_cases hz : z = a
    · simp [hz]
    · by_cases hy : y = a
      · simp [hy, hz] at hyz
      · by_cases hx : x = a
        · simp [hx, hy, hz] at hxy
        · simp [hx, hy, hz] at *
          exact p.trans x y z hxy hyz
}

#check @makeTop
#check @makeBot

---

## 2. Axiomes d'Arrow

Le theoreme d'Arrow repose sur trois axiomes qu'on pourrait considerer comme "raisonnables" pour un systeme de vote.

### 2.1 Pareto faible (P)

Si TOUS les individus preferent strictement x a y, alors la societe prefere strictement x a y.

In [ ]:
-- Axiome de Pareto faible
def WeakPareto {I A : Type} (swf : SocialWelfareFunction I A) : Prop :=
  ∀ (prefs : Profile I A) (x y : A),
    (∀ i : I, (prefs i).strict x y) →
    (swf.f prefs).strict x y

#check @WeakPareto

### 2.2 Independance des alternatives non pertinentes (IIA)

La preference sociale entre x et y ne depend QUE des preferences individuelles entre x et y.

In [ ]:
-- Independance des Alternatives Irrelevantes (IIA)
def IIA {I A : Type} (swf : SocialWelfareFunction I A) : Prop :=
  ∀ (prefs prefs2 : Profile I A) (x y : A),
    (∀ i : I, (prefs i).R x y ↔ (prefs2 i).R x y) →
    (∀ i : I, (prefs i).R y x ↔ (prefs2 i).R y x) →
    ((swf.f prefs).R x y ↔ (swf.f prefs2).R x y)

#check @IIA

### 2.3 Non-dictature

Il n'existe PAS d'individu d tel que pour TOUT profil et TOUTE paire, si d prefere strictement x a y, alors la societe aussi.

In [ ]:
-- Dictateur : un individu dont la preference stricte est toujours suivie
def IsDictator {I A : Type} (swf : SocialWelfareFunction I A) (d : I) : Prop :=
  ∀ (prefs : Profile I A) (x y : A),
    (prefs d).strict x y → (swf.f prefs).strict x y

-- Non-dictature : il n'existe pas de dictateur
def NonDictatorial {I A : Type} (swf : SocialWelfareFunction I A) : Prop :=
  ¬ ∃ d : I, IsDictator swf d

#check @IsDictator
#check @NonDictatorial

---

## 3. Theoreme d'Arrow

### 3.1 Enonce

**Theoreme d'Arrow (1951)** : S'il y a au moins 3 alternatives et au moins 2 individus, alors toute fonction de bien-etre social satisfaisant Pareto faible et IIA est dictatoriale.

$$\text{Pareto} \land \text{IIA} \Rightarrow \neg\text{NonDictatorial}$$

In [ ]:
-- Theoreme d'Arrow : toute SWF satisfaisant Pareto et IIA est dictatoriale
-- Version sketch (preuve complete: social_choice_lean/SocialChoice/Arrow.lean, theorem arrow)
-- Preuve Geanakoplos 2005: extremal_lemma -> pivot_exists -> pivot_is_dictator_except_b
--            -> partial_dictator_is_full_dictator -> arrow (0 sorry, ~950 lignes)

theorem arrow_impossibility_sketch 
    {I A : Type} 
    [DecidableEq A]
    [Inhabited I]
    [Inhabited A]
    (swf : SocialWelfareFunction I A)
    (h_pareto : WeakPareto swf)
    (h_iia : IIA swf)
    -- Hypothese de cardinalite : au moins 3 alternatives
    (h_three_alts : ∃ a b c : A, a ≠ b ∧ b ≠ c ∧ a ≠ c) :
    -- Conclusion : il existe un dictateur
    ∃ d : I, IsDictator swf d := by
  sorry

#check @arrow_impossibility_sketch

### 3.2 Lemmes cles de la preuve

La preuve classique du theoreme d'Arrow procede en plusieurs etapes.

In [ ]:
-- Lemme 1 : Extremal Lemma
-- Si tous les individus placent a en position extreme, la preference sociale aussi

def isTop {A : Type} (p : Preference A) (a : A) : Prop :=
  ∀ x : A, p.R a x

def isBot {A : Type} (p : Preference A) (a : A) : Prop :=
  ∀ x : A, p.R x a

theorem extremal_lemma 
    {I A : Type} [DecidableEq A]
    (swf : SocialWelfareFunction I A)
    (h_pareto : WeakPareto swf)
    (h_iia : IIA swf)
    (prefs : Profile I A) (a : A)
    (h_extreme : ∀ i, isTop (prefs i) a ∨ isBot (prefs i) a) :
    isTop (swf.f prefs) a ∨ isBot (swf.f prefs) a := by
  sorry

#check @extremal_lemma

Le **Lemme Extremal** est la première étape de la preuve : il montre que si tous les individus placent une alternative en position extrême (première ou dernière), alors la société fait de même.

Le lemme suivant introduit la notion d'**ensemble décisif** : un groupe qui peut "imposer" sa préférence sur une paire d'alternatives.

In [ ]:
-- Lemme 2 : Ensemble decisif
-- Un groupe G est decisif pour (x, y) si quand tous dans G preferent x a y,
-- la societe prefere aussi x a y

def IsDecisivePred {I A : Type} (swf : SocialWelfareFunction I A) 
    (G : I → Prop) (x y : A) : Prop :=
  ∀ prefs : Profile I A,
    (∀ i : I, G i → (prefs i).strict x y) →
    (swf.f prefs).strict x y

-- Par Pareto, tous les individus ensemble sont decisifs
theorem all_decisive_pred {I A : Type} 
    (swf : SocialWelfareFunction I A)
    (h_pareto : WeakPareto swf)
    (x y : A) : 
    IsDecisivePred swf (fun _ => True) x y := by
  intro prefs h_all
  apply h_pareto
  intro i
  exact h_all i True.intro

#check @all_decisive_pred

---

## 4. Theoreme de Sen

### 4.1 Le paradoxe liberal

Le **theoreme de Sen** (1970) montre un conflit entre **liberte individuelle** et **efficacite Pareto**.

### 4.2 Liberte minimale

In [ ]:
-- Liberte minimale (Minimal Liberalism) - version bidirectionnelle (Sen 1970)
-- Chaque individu est "decisif" sur au moins une paire d'alternatives,
-- dans les DEUX directions : s'il prefere x a y, la societe aussi,
-- et s'il prefere y a x, la societe aussi.
-- Cette bidirectionnalite est essentielle pour le theoreme d'impossibilite.

def MinimalLiberalism {I A : Type} (swf : SocialWelfareFunction I A) : Prop :=
  ∀ i : I, ∃ x y : A, x ≠ y ∧
    (∀ prefs : Profile I A,
      (prefs i).strict x y → (swf.f prefs).strict x y) ∧
    (∀ prefs : Profile I A,
      (prefs i).strict y x → (swf.f prefs).strict y x)

#check @MinimalLiberalism

La **Liberte Minimale** est une condition tres faible : elle demande seulement que chaque individu ait le controle sur AU MOINS une paire d'alternatives (par exemple, ce qu'il lit dans sa sphere privee).

**Bidirectionnalite** : la definition exige que l'individu soit decisif dans les deux sens — s'il prefere x a y, la societe prefere x a y, ET s'il prefere y a x, la societe prefere y a x. C'est la formulation originale de Sen (1970), et elle est essentielle pour construire le cycle dans la preuve du theoreme d'impossibilite.

Le theoreme de Sen montre que meme cette condition minimale entre en conflit avec l'efficacite Pareto.

> **Preuve complete** : la formalisation complete du theoreme de Sen avec 0 `sorry` se trouve dans le projet Lake `social_choice_lean/` (fichiers `Sen.lean` et `Arrow.lean`).

In [ ]:
-- Theoreme de Sen : impossibilite de satisfaire simultanement Pareto et Liberte
-- Pareto + Liberalisme minimal sont contradictoires
-- Preuve complete: social_choice_lean/SocialChoice/Sen.lean, theorem sen_impossibility
-- (0 sorry, ~300 lignes, construction de profil avec cycle social)

theorem sen_impossibility_sketch
    {I A : Type}
    [DecidableEq A]
    [Inhabited I] [Inhabited A]
    (swf : SocialWelfareFunction I A)
    (h_pareto : WeakPareto swf)
    (h_liberal : MinimalLiberalism swf)
    -- Hypotheses de cardinalite necessaires pour le theoreme
    (h_two_voters : ∃ i j : I, i ≠ j)
    (h_three_alts : ∃ a b c : A, a ≠ b ∧ b ≠ c ∧ a ≠ c) :
    -- Pareto et Liberalisme sont incompatibles
    False := by
  sorry

#check @sen_impossibility_sketch

L'exemple de **Lady Chatterley** est illustre dans le notebook Python compagnon (20b).

---

## 5. Theoreme de l'Electeur Median

### 5.1 Preferences unimodales (Single-peaked)

Le theoreme d'Arrow montre qu'aucune regle de vote parfaite n'existe en general. Cependant, si on **restreint le domaine** a des preferences unimodales, des resultats positifs emergent.

In [ ]:
-- Preferences unimodales (single-peaked)
-- L'espace des alternatives est ordonne (ex: gauche-droite sur [0, 1])

def SinglePeaked {A : Type} (le : A → A → Prop) (p : Preference A) (peak : A) : Prop :=
  -- A gauche du pic : plus on approche du pic, mieux c'est
  (∀ x y : A, le x y → le y peak ∨ y = peak → p.R y x) ∧
  -- A droite du pic : plus on s'eloigne du pic, moins bon c'est
  (∀ x y : A, le peak x ∨ peak = x → le x y → p.R x y)

#check @SinglePeaked

-- Un profil est unimodal si tous les electeurs ont des preferences unimodales
-- Note: On utilise un ordre explicite (le) au lieu de [LinearOrder A] pour eviter Mathlib
def SinglePeakedProfile {I A : Type} (le : A → A → Prop) (prefs : Profile I A) : Prop :=
  ∀ i : I, ∃ peak : A, SinglePeaked le (prefs i) peak

#check @SinglePeakedProfile

### 5.2 Theoreme de l'electeur median

**Theoreme (Black 1948)** : Si tous les electeurs ont des preferences unimodales sur un espace unidimensionnel, alors sous vote majoritaire, l'alternative preferee de l'**electeur median** est le **vainqueur de Condorcet**.

In [ ]:
-- Theoreme de l'electeur median : sous preferences unimodales,
-- il existe un gagnant de Condorcet
-- Note: La preuve complete necessite Fintype + LinearOrder + cardinalite impaire.
-- Le projet social_choice_lean ne contient pas encore cette preuve (a venir).

-- Un gagnant de Condorcet : alternative que tout electeur prefere
-- aux alternatives situees du cote oppose de son pic
-- C'est la propriete structurelle qui rend le gagnant de Condorcet possible
def CondorcetWinner {I A : Type}
    (winner : A) (le : A → A → Prop) (prefs : Profile I A) : Prop :=
  ∀ i : I, ∀ peak alt : A,
    SinglePeaked le (prefs i) peak → alt ≠ winner →
      -- Si le pic est a gauche (ou egal) et alt a droite, l'electeur prefere winner
      ((le peak winner ∨ peak = winner) → le winner alt →
        (prefs i).R winner alt) ∧
      -- Si le pic est a droite (ou egal) et alt a gauche, l'electeur prefere winner
      ((le winner peak ∨ winner = peak) → le alt winner →
        (prefs i).R winner alt)

theorem median_voter_theorem_sketch
    {I A : Type} [Inhabited I]
    (le : A → A → Prop)
    (prefs : Profile I A)
    (h_single_peaked : SinglePeakedProfile le prefs) :
    -- Il existe un gagnant de Condorcet
    -- La preuve complete (construction du median) necessite Fintype
    ∃ winner : A, CondorcetWinner winner le prefs := by
  sorry

#check @median_voter_theorem_sketch

---

## 6. Exercices

Trois exercices d'application directe du cadre formel ci-dessus. Les stubs Lean
compilent **tel quels** (C.1) : remplacez `sorry` / les valeurs `0` par votre code.
Les **indices** guident vers les tactiques utiles (`exact`, `intro`), et chaque exercice
est independant (vous pouvez les faire dans n'importe quel ordre).


In [ ]:
-- EXERCICE 1 : Dictature satisfait Pareto et IIA
-- ============================================
-- Une *dictature* copie les preferences d'un individu d pour toute la societe.
-- Objectif : definir la SWF `dictatorship` et prouver Pareto + IIA.

-- TODO etudiant : Definir dictatorship (SWF qui copie les preferences de d)
-- Indice : le champ f de la SWF est fun prefs => prefs d
def dictatorship {I A : Type} (d : I) : SocialWelfareFunction I A where
  f := fun prefs => prefs d

-- TODO etudiant : Prouver qu'une dictature satisfait Pareto faible
-- Indice : intro prefs x y h_all, puis exact h_all d
theorem dictatorship_pareto {I A : Type} (d : I) :
    WeakPareto (dictatorship d : SocialWelfareFunction I A) := by
  intro prefs x y h_all
  exact h_all d

-- TODO etudiant : Prouver qu'une dictature satisfait IIA
-- Indice : intro p1 p2 x y hR hR', puis exact (hR d)
theorem dictatorship_iia {I A : Type} (d : I) :
    IIA (dictatorship d : SocialWelfareFunction I A) := by
  intro p1 p2 x y hR hR'
  exact (hR d)

#eval "Exercice 1 : dictatorship + Pareto + IIA definis"


In [ ]:
-- EXERCICE 2 : Cycle de Condorcet sur 3 alternatives
-- ===================================================
-- 3 electeurs (0, 1, 2) et 3 alternatives (a, b, c).
--   Electeur 0 : a > b > c
--   Electeur 1 : b > c > a
--   Electeur 2 : c > a > b
-- Question : y a-t-il un vainqueur de Condorcet ? (Reponse : NON, cycle)

-- TODO etudiant : completer les marges (nombre d'electeurs preferant x a y)
-- Indice : compter dans les preferences ci-dessus.
--   a vs b : electeurs 0 et 2 preferent a (2 voix)
--   b vs c : electeurs 0 et 1 preferent b (2 voix)
--   c vs a : electeurs 1 et 2 preferent c (2 voix)

def margin_a_vs_b : Nat := 2  -- TODO etudiant : remplacer par votre calcul
def margin_b_vs_c : Nat := 2  -- TODO etudiant : remplacer par votre calcul
def margin_c_vs_a : Nat := 2  -- TODO etudiant : remplacer par votre calcul

-- Verification : toutes les marges = 2 => cycle parfait (pas de Condorcet)
#eval (margin_a_vs_b, margin_b_vs_c, margin_c_vs_a)
-- Resultat attendu : (2, 2, 2)

-- Extension (optionnelle) : definir une fonction generique margin
-- qui prend deux alternatives et un profil, et retourne le score.
-- TODO etudiant : implementer (indice : utiliser Fin 3 pour les electeurs)

#eval "Exercice 2 : marges pair-a-pair et cycle de Condorcet"


In [ ]:
-- EXERCICE 3 : Preferences unimodales et electeur median
-- ======================================================
-- Theoreme (Black 1948) : sous preferences unimodales, l'electeur median
-- est un vainqueur de Condorcet. On verifie sur un exemple concret.

-- 3 electeurs sur un axe [0, 1] avec pics 0.2, 0.5, 0.8
-- 3 alternatives positionnees : A=0.0, M=0.5, D=1.0
-- Question : qui est le vainqueur de Condorcet ?

-- TODO etudiant : pour chaque electeur, determiner l'ordre de preference
-- Indice : calculer |pic - position| pour chaque alternative.
--   Electeur pic=0.2 : |0.2-0.0|=0.2 < |0.2-0.5|=0.3 < |0.2-1.0|=0.8
--     => A > M > D
--   Electeur pic=0.5 : |0.5-0.0|=0.5 > |0.5-0.5|=0.0 < |0.5-1.0|=0.5
--     => M > A=D
--   Electeur pic=0.8 : |0.8-1.0|=0.2 < |0.8-0.5|=0.3 < |0.8-0.0|=0.8
--     => D > M > A

-- Marge de M vs A : 2/3 preferent M a A (electeurs 0.2 et 0.5)
-- Marge de M vs D : 2/3 preferent M a D (electeurs 0.5 et 0.8)
-- Conclusion : M bat tout le monde => vainqueur de Condorcet = M (median)

def margin_M_vs_A : Nat := 2  -- TODO etudiant : nombre d'electeurs preferant M a A
def margin_M_vs_D : Nat := 2  -- TODO etudiant : nombre d'electeurs preferant M a D

-- Verification : M bat A et M bat D => M = vainqueur de Condorcet
#eval (margin_M_vs_A, margin_M_vs_D)
-- Resultat attendu : (2, 2) => M bat tous => Condorcet

-- Pourquoi M ? L'electeur median (pic=0.5) prefere M. Le theoreme de Black
-- garantit que ce choix est toujours un vainqueur de Condorcet sous unimodalite.
-- Comparez avec l'Exercice 2 : sans unimodalite, il peut ne pas y avoir de Condorcet.

#eval "Exercice 3 : preferences unimodales -> median = Condorcet"


---

## Synthese

Ce notebook a formalise en Lean 4 les **definitions et axiomes** du choix social,
esquisse les **deux theoremes fondateurs** (Arrow, Sen) et illustre le **theoreme de
l'electeur median**. Les exercices ont permis de manipuler concretement :

1. Une SWF dictatoriale (cas pathologique satisfaisant Pareto+IIA)
2. Un cycle de Condorcet sur un profil 3x3 (pas de vainqueur)
3. Le role cle des preferences unimodales pour garantir un vainqueur de Condorcet

| Concept | Definition Lean | Cas d'usage |
|---------|----------------|-------------|
| Preference | `Preference A` | relation binaire faible + complete + transitive |
| Profil | `Profile I A = I -> Preference A` | preferences de chaque individu |
| SWF | `SocialWelfareFunction I A` | agrege profiles en preference sociale |
| Pareto | `WeakPareto swf` | unanimite respectee |
| IIA | `IIA swf` | independance options non pertinentes |
| Dictateur | `IsDictator swf d` | d impose ses preferences |

**Theoreme d'Arrow (1951)** : avec 3+ alternatives et 2+ individus, toute SWF
satisfaisant Pareto + IIA est dictatoriale. Prouve formellement dans
`MyIA.AI.Notebooks/GameTheory/social_choice_lean/SocialChoice/Arrow.lean`.

**Theoreme de Sen (1970)** : Pareto + liberalisme minimal sont incompatibles.
Prouve dans `Sen.lean` (ligne 66). Le paradoxe de Lady Chatterley illustre ce
resultat dans le notebook Python compagnon `03-Voting-Methods.ipynb`.

L'**annexe qui suit** presente le framework de Dominik Peters (15+ regles de vote,
theoreme de Gibbard-Satterthwaite, Split Cycle), formalise dans
`social_choice_lean_peters/`.


---

# Annexe : Tour de SocialChoiceLean (Dominik Peters)

Cette section presente la librairie [SocialChoiceLean](https://github.com/dominikpeters/social-choice-lean) de Dominik Peters,
qui formalise en Lean 4 (avec Mathlib) de nombreuses regles de vote et theoremes d'impossibilite.
Elle complete les preuves du port  etendues dans ce notebook.


---

## 1. Cadre Formel

DominikPeters utilise un cadre base sur les **preferences strictes** (ordres lineaires stricts), ou chaque electeur a un classement sans ex aequo. Ce choix simplifie les definitions des marges et des axiomes par rapport aux preferences faibles (16b).

In [ ]:
-- Simplified framework inspired by DominikPeters/SocialChoiceLean
-- Full formalization (with Mathlib): social_choice_lean_peters/PetersTour.lean

-- Strict preference: total strict order
-- Replaces Mathlib's LinearOrder for kernel compatibility
structure StrictPref (A : Type) where
  lt : A → A → Prop
  irrefl : ∀ x, ¬ lt x x
  trans : ∀ x y z, lt x y → lt y z → lt x z
  conn : ∀ x y, x ≠ y → lt x y ∨ lt y x

-- Voting profile: each voter has a strict preference
def VotingProfile (V A : Type) := V → StrictPref A

-- Voting rule: maps profile to a set of winners (as predicate)
def VotingRule (V A : Type) := VotingProfile V A → (A → Prop)

-- A rule is resolute if it always selects exactly one winner
def IsResolute {V A : Type} (f : VotingRule V A) : Prop :=
  ∀ P, ∃ c : A, f P c ∧ ∀ d, f P d → d = c

#check @StrictPref
#check @VotingProfile
#check @VotingRule
#check @IsResolute

### Interpretation

| Concept | Peters (Mathlib) | Ce notebook (simplifie) |
|---------|-------------------|-------------------------|
| Preference stricte | `LinearOrder A` (classe) | `StrictPref A` (structure) |
| Profil | `Profile V A [Fintype V]` | `VotingProfile V A = V -> StrictPref A` |
| Regle | `VotingRule` (polymorphe Fintype) | `VotingRule V A = Profile -> (A -> Prop)` |
| Resolutude | `Resolute f` | `IsResolute f` |

Dans Peters, `LinearOrder` fournit `lt` automatiquement. Ici, on le definit explicitement avec `irrefl`, `trans`, et `conn` (totalite pour les elements distincts).

---

## 2. Marges et Vainqueur de Condorcet

La **marge** d'un candidat sur un autre mesure l'ecart de votes : combien d'electeurs preferent a a b, moins combien preferent b a a. Un **vainqueur de Condorcet** bat tous les autres par marge positive.

In [ ]:
-- Margin function: pairwise difference
-- In Peters: (Finset.univ.filter (fun v => Prefers P v a b)).card
--   minus the reverse. Here: abstract signature.
abbrev MarginFun (A : Type) := A → A → Int

-- Condorcet winner: beats every other by positive margin
def IsCondorcetWinner {A : Type} (m : MarginFun A) (c : A) : Prop :=
  ∀ d, d ≠ c → m c d > 0

-- Condorcet consistency: the rule selects the Condorcet winner uniquely
def CondorcetConsistent {V A : Type} (f : VotingRule V A)
    (margin : VotingProfile V A → MarginFun A) : Prop :=
  ∀ P c, IsCondorcetWinner (margin P) c →
    f P c ∧ ∀ d, d ≠ c → ¬ f P d

#check @MarginFun
#check @IsCondorcetWinner
#check @CondorcetConsistent

### Interpretation

La **marge** est l'outil central du framework de Peters. Elle permet de definir :

- **Condorcet winner** : candidat avec marge positive sur tous les autres
- **Condorcet consistency** : la regle Selectionne toujours le Condorcet winner

Le concept de marge est intuitif : si 7 electeurs preferent A a B et 3 preferent B a A, la marge de A sur B est +4. Un Condorcet winner a une marge positive contre chaque adversaire.

> **Note technique** : dans la formalisation complete, la marge utilise `Finset.card` (nombre d'electeurs preferant a a b). Notre version simplifiee prend la marge comme fonction abstraite.

---

## 3. Axiomes de Vote

DominikPeters definit et utilise 15+ axiomes pour classifier les regles de vote. Chaque axiome exprime une propriete desirable : equite, robustesse, non-manipulabilite.

In [ ]:
-- Pareto Efficiency: if all prefer c to d, d cannot win
def SatisfiesPareto {V A : Type} (f : VotingRule V A) : Prop :=
  ∀ P c d, (∀ v, (P v).lt c d) → ¬ f P d

-- Unanimity: if all rank c first, c must win
def SatisfiesUnanimity {V A : Type} (f : VotingRule V A) : Prop :=
  ∀ P c, (∀ v d, d ≠ c → (P v).lt c d) → f P c

-- Strategyproof (resolute): no voter can improve outcome by misreporting
-- The voter's true preferences are P v, but they report 'fake' instead
def IsStrategyproof {V A : Type} [DecidableEq A] (f : VotingRule V A) : Prop :=
  ∀ (P : VotingProfile V A) (v : V) (fake : StrictPref A),
    let P' := fun w : V => if w = v then fake else P w
    ¬ ∃ cr cf : A,
      f P cr ∧ f P' cf ∧ (P v).lt cf cr

#check @SatisfiesPareto
#check @SatisfiesUnanimity
#check @IsStrategyproof

### Interpretation des axiomes

| Axiome | Intuition | Exigence |
|--------|-----------|----------|
| **Pareto** | Unanimite : si tous preferent c a d, d est exclu | Minimal pour toute regle raisonnable |
| **Unanimite** | Si tous classent c premier, c doit gagner | Plus fort que Pareto |
| **Non-manipulabilite** | Aucun electeur ne peut ameliorer le resultat en mentant | Equilibre strategique |

Ces trois axiomes semblent naturels et faibles. Pourtant, le theoreme de Gibbard-Satterthwaite montre qu'ils sont **incompatibles** avec la non-dictature pour >= 3 candidats.

---

## 4. Theoreme de Gibbard-Satterthwaite

Le resultat central de la theorie du choix social : toute regle de vote resolutive, unanime et non-manipulable (pour >= 3 candidats) doit etre une dictature. Ce theoreme montre que **toute election est manipulable** (sauf dictature).

La preuve formelle dans Peters fait 5 fichiers Lean (BaseCase + Common + InductionStepCase1 + InductionStepCase2 + Main), avec une induction forte sur le nombre d'electeurs.

In [ ]:
-- Dictatorial (for resolute rules): one voter's preference determines the winner
def IsDictatorialRule {V A : Type} (f : VotingRule V A) : Prop :=
  ∃ d : V, ∀ P c, f P c → ∀ d' : A, d' ≠ c → (P d).lt c d'

/-- **Gibbard-Satterthwaite (1973/1975)**
    Toute regle resolutive, unanime et non-manipulable (>= 3 candidats)
    est dictatoriale.

    Preuve par induction forte sur le nombre d'electeurs :
    - Base (1 electeur) : trivialement dictatorial
    - Induction : clonage d'un electeur, application de l'hypothese de recurrence
    - Full proof: SocialChoice.Impossibilities.GibbardSatterthwaite -/
theorem gibbard_satterthwaite_sketch
    {V A : Type} [DecidableEq A] [Inhabited V] [Inhabited A]
    (f : VotingRule V A)
    (hf_res : IsResolute f)
    (hf_unan : SatisfiesUnanimity f)
    (hf_sp : IsStrategyproof f)
    (h_three : ∃ a b c : A, a ≠ b ∧ b ≠ c ∧ a ≠ c) :
    IsDictatorialRule f := by
  sorry

#check @gibbard_satterthwaite_sketch

### Interpretation

Le theoreme de Gibbard-Satterthwaite a des implications profondes :

1. **Toute election est manipulable** : si la regle n'est pas une dictature, il existe toujours une situation ou un electeur a interet a mentir sur ses preferences
2. **La non-manipulabilite est une chimere** : meme les regles les plus sophistiquees (IRV, Borda, Copeland...) sont manipulables
3. **Le choix du systeme de vote est un compromis** : on choisit quelles proprietes sacrifier

### Structure de la preuve (Peters)

La preuve formelle suit l'approche classique par contraposition :
1. On suppose la regle resolutive, unanime, non-manipulable et non-dictatoriale
2. On montre l'existence d'un "pivot" (electeur dont le vote est critique)
3. Par induction sur le nombre d'electeurs, on construit une contradiction

> **Preuve complete** : `SocialChoice.Impossibilities.GibbardSatterthwaite.Main` dans `social_choice_lean_peters/`

---

## 5. Impossibilites de Condorcet

DominikPeters formalise 4 theoremes d'impossibilite lies a Condorcet. Chacun montre que la coherence de Condorcet entre en conflit avec un autre axiome naturel :

1. **Condorcet + Participation** => Impossible (Moulin 1988)
2. **Condorcet + Renforcement** => Impossible (Young 1975)
3. **Condorcet + Non-manipulabilite** => Impossible
4. **Anonymat + Neutralite + Resolutude** => Impossible (nombre pair)

In [ ]:
-- Condorcet impossibilities: Condorcet consistency + natural axiom => contradiction
-- All formalized in SocialChoice.Impossibilities.* (Peters project)

-- Key result: if a rule is Condorcet consistent, resolute and strategyproof,
-- it cannot exist (for >= 3 alternatives)
theorem condorcet_strategyproof_impossible_sketch
    {V A : Type} [DecidableEq A]
    (f : VotingRule V A)
    (margin : VotingProfile V A → MarginFun A)
    (hf_res : IsResolute f)
    (hf_cc : CondorcetConsistent f margin)
    (hf_sp : IsStrategyproof f)
    (h_three : ∃ a b c : A, a ≠ b ∧ b ≠ c ∧ a ≠ c) :
    False := by
  sorry

-- No-show paradox: adding voters who rank c first can make c lose
-- Formalized in SocialChoice.Impossibilities.CondorcetParticipation
-- (Moulin 1988)

-- Reinforcement: if disjoint electorates agree, the union should agree
-- Incompatible with Condorcet consistency
-- Formalized in SocialChoice.Impossibilities.CondorcetReinforcement
-- (Young 1975)

#check @condorcet_strategyproof_impossible_sketch

### Interpretation des impossibilites de Condorcet

| Impossibilite | Axiomes | Resultat | Reference |
|---------------|---------|----------|-----------|
| Moulin 1988 | Condorcet + Participation | No-show paradox | `CondorcetParticipation` |
| Young 1975 | Condorcet + Renforcement | Incoherence inter-groupes | `CondorcetReinforcement` |
| General | Condorcet + Strategyproof | Manipulable | `CondorcetStrategyproofness` |
| ANR | Anonymat + Neutralite + Resolutude | Impossible (pair) | `AnonymousNeutralResolute` |

Le **no-show paradox** (Moulin 1988) est particulierement frappant : ajouter des electeurs qui classent votre candidat en tete peut le faire perdre ! Cela signifie que ne pas voter peut etre une meilleure strategie que de voter.

Ces resultats montrent que la **coherence de Condorcet** est une condition forte qui exclut de nombreuses proprietes desiderables.

---

## 6. Split Cycle (Holliday & Pacuit 2023)

La regle Split Cycle est un cas particulier dans la theorie du choix social : c'est la regle la plus fine qui soit coherente avec Condorcet et acyclique. Elle resout le probleme des cycles de Condorcet en "affaiblissant" les marges dans les cycles.

In [ ]:
-- Split Cycle (Holliday & Pacuit 2023)
-- The finest Condorcet-consistent, acyclic voting rule

-- Simplified: x defeats y if margin(x,y) > 0
-- and no blocking cycle exists (simplified to 3-cycles here)
-- In Peters: arbitrary-length cycles via List A

variable {A : Type}

-- Blocking cycle (simplified to 3-cycles)
-- A 3-cycle z -> x -> y -> z blocks x -> y if margins are >= margin(x,y)
def HasNoBlockingCycle (m : MarginFun A) (x y : A) : Prop :=
  ¬ ∃ z : A, m z x ≥ m x y ∧ m y z ≥ m x y ∧ z ≠ x ∧ z ≠ y

-- Split Cycle defeat: positive margin + no blocking cycle
def SCSimpleDefeat (m : MarginFun A) (x y : A) : Prop :=
  m x y > 0 ∧ HasNoBlockingCycle m x y

-- Split Cycle winners: undefeated alternatives
def SplitCycleWinners (m : MarginFun A) : A → Prop :=
  fun x => ∀ y, ¬ SCSimpleDefeat m y x

#check @HasNoBlockingCycle
#check @SCSimpleDefeat
#check @SplitCycleWinners

### Interpretation de Split Cycle

L'idee cle de Split Cycle est d'**affaiblir les relations de defaite** : on ne retient la defaite de x sur y que si aucune marge dans un cycle contenant x et y n'est >= la marge de x sur y.

**Intuition** : si la marge de x sur y est 3, mais qu'il existe un cycle x -> y -> z -> x avec des marges >= 3, alors la defaite de x sur y est "bloquee" par le cycle. On ne peut pas affirmer que x bat y sans affirmer simultanement que y bat z et z bat x.

### Axiomes verifiees pour Split Cycle (Peters)

| Axiome | Statut | Fichier |
|--------|--------|---------|
| Condorcet Consistency | Prouve | `SplitCycle/Condorcet.lean` |
| Monotonicite | Prouve | `SplitCycle/Monotonicity.lean` |
| Pareto | Prouve | `SplitCycle/Pareto.lean` |
| Neutrite | Prouve | `SplitCycle/Neutrality.lean` |
| Smith | Prouve | `SplitCycle/Smith.lean` |
| Clones | Prouve | `SplitCycle/Clones.lean` |
| Independance | Prouve | `SplitCycle/Independence.lean` |
| Renversement | Prouve | `SplitCycle/Reversal.lean` |

Split Cycle est unique : c'est la regle la plus fine satisfaisant Condorcet + acyclicite + une autre propriete naturelle (voir Holliday & Pacuit 2023 pour les details).

---

## 7. Regles de Vote Formalisees

DominikPeters verifie systematiquement les axiomes pour 15+ regles de vote, grace au meta-framework `@[scAxiom]` / `@[scRule]`.

### Tableau comparatif des regles

| Regle | Axiomes verifiees |
|-------|-------------------|
| **Split Cycle** | Condorcet, Monotonicity, Pareto, Neutrality, Smith, Clones, Reversal, Independence |
| **Schulze** | Condorcet, Monotonicity, Neutrality, Smith, Clones, Reversal, Transitivity |
| **Black** | Condorcet, Monotonicity, Neutrality, Pareto, Reversal, Smith |
| **Copeland** | Condorcet, Monotonicity, Neutrality, Pareto, Reversal, Smith |
| **Minimax** | Condorcet, Monotonicity, Neutrality, Pareto, Reversal, Majority |
| **Borda** | Monotonicity, Neutrality, Pareto, Participation, Reinforcement |
| **Plurality** | Monotonicity, Majority |
| **IRV** | CondorcetLoser, Majority, MutualMajority, Clones |
| **Baldwin** | Condorcet, CondorcetLoser, Smith |
| **Coombs** | Condorcet, CondorcetLoser, Majority |
| **Top Cycle** | Condorcet, Monotonicity, Smith, MutualMajority |
| **Uncovered Set** | Condorcet, Monotonicity, Neutrality, Smith, Clones |

**12 regles, 14 axiomes distincts**. Chaque verification est une preuve formelle en Lean 4.

---

## 8. Theoreme de Duggan-Schwartz

Le theoreme de Duggan-Schwartz etend Gibbard-Satterthwaite aux **regles multi-gagnants** (qui peuvent selectionner plusieurs candidats). Il utilise deux notions de non-manipulabilite : optimiste et pessimiste.

In [ ]:
-- Duggan-Schwartz: extension to multi-winner rules

-- Optimist strategyproof: can't improve best possible outcome
def IsOptimistSP {V A : Type} [DecidableEq A] (f : VotingRule V A) : Prop :=
  ∀ (P : VotingProfile V A) (v : V) (fake : StrictPref A),
    let P' := fun w : V => if w = v then fake else P w
    ¬ ∃ y, f P' y ∧ ∀ x, f P x → (P v).lt y x

-- Pessimist strategyproof: can't improve worst possible outcome
def IsPessimistSP {V A : Type} [DecidableEq A] (f : VotingRule V A) : Prop :=
  ∀ (P : VotingProfile V A) (v : V) (fake : StrictPref A),
    let P' := fun w : V => if w = v then fake else P w
    ¬ ∃ x, f P x ∧ ∀ y, f P' y → (P v).lt y x

/-- **Duggan-Schwartz (2000)**: A non-trivial, surjective voting rule
    satisfying both optimist and pessimist strategyproofness
    has a "nominating set" (coalition of <= 3 voters controlling outcome).
    Full proof: SocialChoice.Impossibilities.DugganSchwartz -/
theorem duggan_schwartz_sketch
    {V A : Type} [DecidableEq A] [Inhabited V] [Inhabited A]
    (f : VotingRule V A)
    (hf_opt : IsOptimistSP f)
    (hf_pess : IsPessimistSP f)
    (h_three : ∃ a b c : A, a ≠ b ∧ b ≠ c ∧ a ≠ c) :
    ∃ (G : V → Prop), True := by sorry

#check @IsOptimistSP
#check @IsPessimistSP
#check @duggan_schwartz_sketch

### Interpretation de Duggan-Schwartz

Le theoreme de Duggan-Schwartz montre que meme les regles multi-gagnants sont vulnerables :

- **Manipulation optimiste** : un electeur peut ameliorer le **meilleur** candidat elu en mentant
- **Manipulation pessimiste** : un electeur peut ameliorer le **pire** candidat elu en mentant
- Si une regle est non-triviale et surjective, elle doit avoir un "nominating set" (petite coalition controlant le resultat)

Ce theoreme est plus general que Gibbard-Satterthwaite : il s'applique meme aux regles qui peuvent eligir plusieurs candidats.

---

**Navigation** : [<< 01-Arrow-Impossibility-Theorem.ipynb](01-Arrow-Impossibility-Theorem.ipynb) | [03-Voting-Methods.ipynb >>](03-Voting-Methods.ipynb) | [Index](../README.md)
